setup


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import torchvision.transforms.functional as F
sys.modules['torchvision.transforms.functional_tensor'] = F

!pip install pytorchvideo -q
!pip install wandb -q

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os

BASE = '/content/drive/MyDrive/Ketastasia/data'
print("Setup complete")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete


manifest-ის ჩატვირთვა

In [2]:
manifest = pd.read_csv(f'{BASE}/combined_cached_final.csv')

classes = sorted(manifest['label'].unique())
label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {i: c for c, i in label2idx.items()}
NUM_CLASSES = len(classes)

print(f"{NUM_CLASSES} classes: {classes}")
print(manifest['split'].value_counts())

train_df = manifest[manifest['split'] == 'train'].reset_index(drop=True)
val_df   = manifest[manifest['split'] == 'val'].reset_index(drop=True)
# test_df-ს ჯერ არ ვეხებით

print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}")

22 classes: ['bench_press', 'bicep_curl', 'chest_fly', 'clean_and_jerk', 'deadlift', 'hammer_curl', 'hip_thrust', 'jump_rope', 'jumping_jacks', 'lat_pulldown', 'lateral_raise', 'leg_extension', 'leg_raises', 'pullup', 'pushup', 'russian_twist', 'shoulder_press', 'situp', 'squat', 't_bar_row', 'tricep_dips', 'tricep_pushdown']
split
train    1083
test      362
val       361
Name: count, dtype: int64

Train: 1083, Val: 361


In [3]:
if not os.path.exists('/content/frame_cache'):
    print("Extracting frame cache...")
    !tar -xzf '{BASE}/frame_cache.tar.gz' -C /content
else:
    print("Frame cache already present")

Extracting frame cache...


 PyTorch Dataset (Cached Clips)

In [4]:
from torch.utils.data import Dataset, DataLoader

class CachedClipDataset(Dataset):
    def __init__(self, df, label2idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        frames = np.load(row['cache_path'])                                   # (32,256,256,3) uint8
        clip = torch.from_numpy(frames).permute(3, 0, 1, 2).float() / 255.0    # (C,T,H,W)
        if self.transform:
            clip = self.transform(clip)
        label = self.label2idx[row['label']]
        return clip, label

print("Dataset class defined")

Dataset class defined


Class weights (Effective Number of Samples)

In [5]:
beta = 0.999
class_counts = train_df['label'].value_counts().reindex(classes).values
effective_num = 1.0 - np.power(beta, class_counts)
weights = (1.0 - beta) / effective_num
weights = weights / weights.sum() * NUM_CLASSES

class_weights = torch.tensor(weights, dtype=torch.float32)
for c, w in zip(classes, weights):
    print(f"{c:25s} w={w:.3f}")

bench_press               w=0.150
bicep_curl                w=0.583
chest_fly                 w=1.156
clean_and_jerk            w=0.399
deadlift                  w=0.803
hammer_curl               w=1.884
hip_thrust                w=1.884
jump_rope                 w=0.415
jumping_jacks             w=0.323
lat_pulldown              w=0.698
lateral_raise             w=0.947
leg_extension             w=1.385
leg_raises                w=1.596
pullup                    w=0.166
pushup                    w=0.139
russian_twist             w=2.587
shoulder_press            w=2.072
situp                     w=0.348
squat                     w=0.144
t_bar_row                 w=1.596
tricep_dips               w=2.072
tricep_pushdown           w=0.655


Evaluation Function

In [6]:
from sklearn.metrics import f1_score, classification_report

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def evaluate(model, loader, device, criterion):
    model.eval()
    all_preds, all_labels = [], []
    val_loss = 0
    with torch.no_grad():
        for clips, labels in loader:
            clips, labels = clips.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(clips)
                loss = criterion(outputs, labels)
            val_loss += loss.item()
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    report = classification_report(all_labels, all_preds, target_names=classes, zero_division=0)
    return val_loss / len(loader), acc, macro_f1, report

print(f"Using device: {device}")

Using device: cuda


WandB login

In [8]:
import wandb
from google.colab import userdata

wandb_api_key = userdata.get('WANDB_API_KEY_1')
wandb.login(key=wandb_api_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akeke23 (akeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
from pytorchvideo.transforms import UniformTemporalSubsample, ShortSideScale
from torchvision.transforms import Compose, CenterCrop, RandomCrop, RandomHorizontalFlip

X3D_S_NUM_FRAMES = 13
X3D_S_SIDE_SIZE = 182
X3D_S_CROP = 182
MEAN = [0.45, 0.45, 0.45]
STD  = [0.225, 0.225, 0.225]

def normalize(x):
    mean = torch.tensor(MEAN).view(3,1,1,1)
    std  = torch.tensor(STD).view(3,1,1,1)
    return (x - mean) / std

train_transform_s = Compose([
    UniformTemporalSubsample(X3D_S_NUM_FRAMES),
    normalize,
    ShortSideScale(size=X3D_S_SIDE_SIZE),
    RandomCrop(X3D_S_CROP),
    RandomHorizontalFlip(p=0.5),
])
val_transform_s = Compose([
    UniformTemporalSubsample(X3D_S_NUM_FRAMES),
    normalize,
    ShortSideScale(size=X3D_S_SIDE_SIZE),
    CenterCrop(X3D_S_CROP),
])

train_ds_s = CachedClipDataset(train_df, label2idx, transform=train_transform_s)
val_ds_s   = CachedClipDataset(val_df, label2idx, transform=val_transform_s)

train_loader_s = DataLoader(train_ds_s, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader_s   = DataLoader(val_ds_s, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader_s)}, Val batches: {len(val_loader_s)}")

Train batches: 136, Val batches: 46


In [10]:
sample_clip, sample_label = train_ds_s[0]
print(f"Clip shape: {sample_clip.shape}")
print(f"Expected: (3, {X3D_S_NUM_FRAMES}, {X3D_S_CROP}, {X3D_S_CROP})")
assert sample_clip.shape == (3, X3D_S_NUM_FRAMES, X3D_S_CROP, X3D_S_CROP), "Shape mismatch!"
print("Shape verified correctly")

Clip shape: torch.Size([3, 13, 182, 182])
Expected: (3, 13, 182, 182)
Shape verified correctly


In [15]:
model_s = torch.hub.load('facebookresearch/pytorchvideo', 'x3d_s', pretrained=True)
in_features = model_s.blocks[-1].proj.in_features
model_s.blocks[-1].proj = nn.Linear(in_features, NUM_CLASSES)
model_s = model_s.to(device)

criterion_unweighted = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model_s.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
scaler = torch.cuda.amp.GradScaler()

wandb.init(
    project="ildolcefarniente",
    group="p3_x3d",
    name="X3D-S_baseline",
    config={
        "model": "x3d_s", "lr": 1e-4, "batch_size": 8,
        "num_classes": NUM_CLASSES, "class_weighted": False,
        "num_frames": X3D_S_NUM_FRAMES, "crop_size": X3D_S_CROP
    }
)

best_val_macro_f1 = 0
patience, patience_counter = 10, 0
best_report = ""

for epoch in range(30):
    model_s.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for clips, labels in train_loader_s:
        clips, labels = clips.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model_s(clips)
            loss = criterion_unweighted(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        preds = outputs.argmax(1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total
    val_loss, val_acc, val_macro_f1, report = evaluate(model_s, val_loader_s, device, criterion_unweighted)
    scheduler.step()

    gap = train_acc - val_acc

    wandb.log({
        "train_loss": train_loss / len(train_loader_s),
        "train_acc": train_acc,
        "val_loss": val_loss, "val_acc": val_acc, "val_macro_f1": val_macro_f1,
        "overfitting_gap": gap,
        "epoch": epoch + 1
    })
    print(f"Epoch {epoch+1:02d}: train_loss={train_loss/len(train_loader_s):.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_macro_f1:.4f} gap={gap:.4f}")

    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        patience_counter = 0
        best_report = report
        # ინახება პირდაპირ Drive-ზე
        torch.save(model_s.state_dict(), '/content/drive/MyDrive/Ketastasia/models/x3d_s_baseline_best.pt')
        print(f"   -> New best model saved to Drive! (val_macro_f1={val_macro_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

print(f"\nBest val macro-F1: {best_val_macro_f1:.4f}")
print("\n=== Final per-class report (best epoch) ===")
print(best_report)
wandb.finish()

Using cache found in /root/.cache/torch/hub/facebookresearch_pytorchvideo_main
/tmp/ipykernel_1773/3047113673.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


/tmp/ipykernel_1773/3047113673.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_1773/607055086.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 01: train_loss=2.7161 train_acc=0.5060 | val_loss=2.4700 val_acc=0.7424 val_macro_f1=0.2981 gap=-0.2364
   -> New best model saved to Drive! (val_macro_f1=0.2981)


/tmp/ipykernel_1773/3047113673.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_1773/607055086.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 02: train_loss=2.4240 train_acc=0.7673 | val_loss=2.3659 val_acc=0.8338 val_macro_f1=0.4413 gap=-0.0665
   -> New best model saved to Drive! (val_macro_f1=0.4413)


...

/tmp/ipykernel_1773/3047113673.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_1773/607055086.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 29: train_loss=2.1682 train_acc=1.0000 | val_loss=2.1940 val_acc=0.9751 val_macro_f1=0.9454 gap=0.0249


/tmp/ipykernel_1773/3047113673.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_1773/607055086.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 30: train_loss=2.1684 train_acc=0.9991 | val_loss=2.1913 val_acc=0.9834 val_macro_f1=0.9610 gap=0.0157

Best val macro-F1: 0.9726

=== Final per-class report (best epoch) ===
                 precision    recall  f1-score   support

    bench_press       1.00      1.00      1.00        49
     bicep_curl       1.00      1.00      1.00        13
      chest_fly       1.00      1.00      1.00         5
 clean_and_jerk       1.00      1.00      1.00        17
       deadlift       1.00      1.00      1.00        11
    hammer_curl       1.00      1.00      1.00         3
     hip_thrust       1.00      1.00      1.00         3
      jump_rope       1.00      1.00      1.00        17
  jumping_jacks       1.00      1.00      1.00        23
   lat_pulldown       1.00      1.00      1.00         9
  lateral_raise       1.00      1.00      1.00         8
  leg_extension       0.83      1.00      0.91         5
     leg_raises       1.00      1.00      1.00         4
         pullup     

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
overfitting_gap,▁▆▇▇▇▇████████████████████████
train_acc,▁▅▆▆▇▇▇▇▇▇▇███████████████████
train_loss,█▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▄▄▅▆▆▆▇▇▇▇▇▇▇▇▇█▇▇███████████
val_loss,█▅▅▃▃▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_macro_f1,▁▂▃▃▅▅▅▆▆▆▆▆▇▇▆▇▇▇▇▇▇▇████████
epoch,30
overfitting_gap,0.0157
train_acc,0.99908
train_loss,2.16844


In [ ]:
model_s_cw = torch.hub.load('facebookresearch/pytorchvideo', 'x3d_s', pretrained=True)
in_features = model_s_cw.blocks[-1].proj.in_features
model_s_cw.blocks[-1].proj = nn.Linear(in_features, NUM_CLASSES)
model_s_cw = model_s_cw.to(device)

criterion_weighted = nn.CrossEntropyLoss(weight=class_weights.to(device))

optimizer = torch.optim.AdamW(model_s_cw.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
scaler = torch.cuda.amp.GradScaler()

wandb.init(
    project="ildolcefarniente",
    group="p3_x3d",
    name="X3D-S_class_weighted",
    config={
        "model": "x3d_s", "lr": 1e-4, "batch_size": 8,
        "num_classes": NUM_CLASSES, "class_weighted": True,
        "num_frames": X3D_S_NUM_FRAMES, "crop_size": X3D_S_CROP
    }
)

best_val_macro_f1 = 0
patience, patience_counter = 10, 0
best_report = ""

for epoch in range(30):
    model_s_cw.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for clips, labels in train_loader_s:
        clips, labels = clips.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model_s_cw(clips)
            loss = criterion_weighted(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        preds = outputs.argmax(1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total
    val_loss, val_acc, val_macro_f1, report = evaluate(model_s_cw, val_loader_s, device, criterion_weighted)
    scheduler.step()

    gap = train_acc - val_acc

    wandb.log({
        "train_loss": train_loss / len(train_loader_s),
        "train_acc": train_acc,
        "val_loss": val_loss, "val_acc": val_acc, "val_macro_f1": val_macro_f1,
        "overfitting_gap": gap,
        "epoch": epoch + 1
    })
    print(f"Epoch {epoch+1:02d}: train_loss={train_loss/len(train_loader_s):.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_macro_f1:.4f} gap={gap:.4f}")

    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        patience_counter = 0
        best_report = report
        torch.save(model_s_cw.state_dict(), '/content/drive/MyDrive/Ketastasia/models/x3d_s_classweighted_best.pt')
        print(f"   ->New best model saved to Drive! (val_macro_f1={val_macro_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

print(f"\nBest val macro-F1: {best_val_macro_f1:.4f}")
print("\n=== Final per-class report (best epoch) ===")
print(best_report)
wandb.finish()

Using cache found in /root/.cache/torch/hub/facebookresearch_pytorchvideo_main
/tmp/ipykernel_1773/280466488.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


/tmp/ipykernel_1773/280466488.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_1773/607055086.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
